In [160]:
user = 'magnu'

In [161]:
import os
import pandas as pd
dir_path = fr"C:\Users\{user}\OneDrive - University of Copenhagen\Project Lanbarcode batch upload\lanebarcode"
file_names = os.listdir(dir_path)
lane_barcodes = pd.DataFrame(file_names)

In [162]:
lane_barcodes.rename(columns={0: 'filename'}, inplace=True)

In [163]:
for row in lane_barcodes.iterrows():
    split = row[1]['filename'].split('.')
    if split[1:] != ['lanebarcode', 'html']:
        raise ValueError(f"Unexpected file format: {row[1]['filename']}")
    else :
        lane_barcodes.at[row[0], 'run'] = split[0]

In [164]:
lane_barcodes['date'] = lane_barcodes['run'].apply(lambda x: x.split('_')[0])
lane_barcodes['date'] = lane_barcodes['date'].apply(lambda x: x[2:] if len(x) == 8 else x)
lane_barcodes['machine'] = lane_barcodes['run'].apply(lambda x: x.split('_')[1])
lane_barcodes['run_no'] = lane_barcodes['run'].apply(lambda x: x.split('_')[2])
lane_barcodes['side'] = lane_barcodes['run'].apply(lambda x: x.split('_')[3][0])
lane_barcodes['flowcell'] = lane_barcodes['run'].apply(lambda x: x.split('_')[3][1:])

In [165]:
assert all(lane_barcodes.iloc[:, 2:].astype(str).map(len).nunique() == 1)

In [166]:
def find_valid_elements(path):
        path_split = path.split('_')
        res = []
        for ele in path_split:
            if len(ele)== 5 and not any(x.islower() for x in ele):
                res.append(ele)
        if len(res) > 0:
            return res
        else:
            return None
lane_barcodes['ttags'] = lane_barcodes.run.apply(find_valid_elements)

In [167]:
lane_barcodes['full_flowcell_id'] = lane_barcodes['side'] + lane_barcodes['flowcell']

In [168]:
helpr_path = fr"C:\Users\{user}\OneDrive - University of Copenhagen\Project Lanbarcode batch upload\Copy of list_all_SeqRuns_2024-2025_edit251113JBT.xlsx"
pool_tags = pd.read_excel(helpr_path)

In [169]:
pool_tags['full_flowcell_id'] = pool_tags['InstrumentSide'] + pool_tags['FlowCellID']
# We don't expect XP runs here
mask = pool_tags['RunSheetID'].apply(lambda x: 'xp' in x.lower() if isinstance(x, str) else x)
assert len(pool_tags[mask]) == 0
mask = pool_tags['full_flowcell_id'].duplicated(keep=False)
pool_tags['duplicate_flowcell'] = mask

In [170]:
# get only unique flowcells that are not xp runs
mask1 = ~pool_tags['full_flowcell_id'].duplicated(keep=False)
clean_pool_tags = pool_tags[mask1]
trash_pool_tags = pool_tags[~mask1]
trash_pool_tags.to_excel(fr"C:\Users\{user}\OneDrive - University of Copenhagen\Project Lanbarcode batch upload\trash_pool_tags.xlsx", index=False)

In [171]:
assert len(set(clean_pool_tags['full_flowcell_id'])) == len(clean_pool_tags)

In [172]:
merged = clean_pool_tags.merge(lane_barcodes, on='full_flowcell_id', how='inner', validate='one_to_many')

In [173]:
assert all(merged['side'] == merged['InstrumentSide']) 
assert not any(merged['filename'].duplicated(keep=False))

In [174]:
correct_files = ['240724_A00706_0869_BH5F2GDSX7_PQFUO_Marina.lanebarcode.html',
'240315_A00706_0844_BH37VGDSX7_ZWT6V_reDMPX.lanebarcode.html',
'20250219_A00706_0914_AHVVLWDSXC.lanebarcode.html',
'240806_A00706_0875_AH5FFKDSX7_77P4L.lanebarcode.html',
'240216_A00706_0826_AH3725DSX7_L8R4I_reDMPX_new.lanebarcode.html']

In [175]:
# Some flowcell IDs occur in multiple lanebarcode filenames. Those are removed here.
mask = merged['full_flowcell_id'].duplicated(keep=False)
mask2 = merged['filename'].isin(correct_files)
merged_no_dups = merged[~mask | mask2]
assert not any(merged_no_dups['full_flowcell_id'].duplicated(keep=False))
assert len(merged[mask]) == (len(merged) - len(merged_no_dups)) + len(correct_files)

In [176]:
extra_files = merged[mask & ~mask2]
extra_files.to_excel(fr"C:\Users\{user}\OneDrive - University of Copenhagen\Project Lanbarcode batch upload\extra_files.xlsx", index=False)

In [177]:
from pathlib import Path
import sys
import os

def find_project_root():
    path = Path(os.getcwd()).resolve()
    while path != path.root:
        if (path / 'very_rootsy_file.txt').exists():
            return path
        path = path.parent
    return None  # Project root not found

project_root = find_project_root()

sys.path.append(str(project_root))

In [178]:
%env PGPASSWORD=5J8DhII0RRsPW1

env: PGPASSWORD=5J8DhII0RRsPW1


In [179]:
from constants.db_connections import ENGINE, ENGINE_READ_ONLY, SQL_ALCH_CONFIG, PSY_CONN
q = 'select * from flowcell'
flowcell_db = pd.read_sql(q, ENGINE_READ_ONLY)
flowcell_db['full_flowcell_id'] = flowcell_db['flowcell_position'] + flowcell_db['flowcell_id']

q = 'select * from top_unknown_seq_barcodes'
tusb_db = pd.read_sql(q, ENGINE_READ_ONLY)

In [180]:
flowcell_to_upload = merged_no_dups
tusb_to_upload = merged_no_dups

In [181]:
mask = flowcell_to_upload['full_flowcell_id'].isin(flowcell_db['full_flowcell_id'])
flowcell_to_upload = flowcell_to_upload[~mask]                                

In [182]:
mask = tusb_to_upload['FlowCellID'].isin(tusb_db['flowcell_id'])
tusb_to_upload = tusb_to_upload[~mask]                                

Get all lanebarcode data to upload into one df

In [183]:
from constants.db_names.names import data
import lane_barcode_parser

fc = []
tpb = []

for file_name in tusb_to_upload.filename:
    file_path = os.path.join(dir_path, file_name)

    tube_tag_col_name = data.flowcell.library_pool_tag()
    read_len_col_name = data.flowcell.read_length()
    seq_machine_col_name = data.flowcell.sequencing_machine()
    seq_date_col_name = data.flowcell.sequencing_date()
    flowcell_pos_col_name = data.flowcell.flowcell_position()
    seq_run_col_name = data.flowcell.sequencing_run()


    # TODO: Make more general:
    sheet = pd.read_html(file_path, thousands=',', decimal='.')

    flowcell_data, top_unknown_barcodes = lane_barcode_parser.parse(df=sheet)

    fc.append(flowcell_data)
    tpb.append(top_unknown_barcodes)

In [184]:
top_unknown_barcodes = pd.concat(tpb, ignore_index=True)
flowcell_data = pd.concat(fc, ignore_index=True)

Get read length, sequencing date, tubetag, seq machine, seq run and flowcell pos merged in

In [185]:
machine_id_map = {
                'A00706': {
                    'name': 'Nova Seq 6000',
                    'rl': '101'
                }
            }
tusb_to_upload.loc[:, 'read_length'] = machine_id_map['A00706']['rl']
tusb_to_upload.loc[:, 'sequencing_machine'] = machine_id_map['A00706']['name']

C:\Users\magnu\AppData\Local\Temp\ipykernel_3396\1588488433.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tusb_to_upload.loc[:, 'read_length'] = machine_id_map['A00706']['rl']
C:\Users\magnu\AppData\Local\Temp\ipykernel_3396\1588488433.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tusb_to_upload.loc[:, 'sequencing_machine'] = machine_id_map['A00706']['name']


In [186]:
tusb_to_upload.loc[:, 'date'] = pd.to_datetime(tusb_to_upload['date'], format='%y%m%d').dt.strftime('%Y-%m-%d')

In [187]:
tusb_to_upload.rename(columns={
    'FlowCellID': data.flowcell.flowcell_id(),
    'run_no': data.flowcell.sequencing_run(),
    'RunDate': data.flowcell.sequencing_date(),
    'InstrumentSide': data.flowcell.flowcell_position(),
    'Tubetag': data.flowcell.library_pool_tag(),
    'filename': 'upload_sheet'
}, inplace=True, errors='raise')

C:\Users\magnu\AppData\Local\Temp\ipykernel_3396\2759310163.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tusb_to_upload.rename(columns={


In [188]:
to_upload_final_subset = tusb_to_upload[['upload_sheet', 'sequencing_run', 'sequencing_date', 'sequencing_machine', 'sequencing_run', 'flowcell_position', 'flowcell_id', 'seqc_tube_tag', 'read_length']]

In [189]:
q = 'select column_name_db, column_name_sheet from name_maps.column_names where table_id = 9'
flowcell_name_maps = pd.read_sql(q, ENGINE_READ_ONLY)
flowcell_name_maps = flowcell_name_maps.dropna()
flowcell_name_maps.index = flowcell_name_maps['column_name_sheet']
flowcell_name_maps.drop(columns=['column_name_sheet'], inplace=True)

In [190]:
q = 'select column_name_db, column_name_sheet from name_maps.column_names where table_id = 18'
tusb_name_maps = pd.read_sql(q, ENGINE_READ_ONLY)
tusb_name_maps = tusb_name_maps.dropna()
tusb_name_maps.index = tusb_name_maps['column_name_sheet']
tusb_name_maps.drop(columns=['column_name_sheet'], inplace=True)

In [191]:
translator_tusb = tusb_name_maps.to_dict(orient='dict')['column_name_db']
translator_fc = flowcell_name_maps.to_dict(orient='dict')['column_name_db']

In [192]:
flowcell_data = flowcell_data.rename(columns=translator_fc)


In [193]:
# to_upload_final_fc = flowcell_data.merge(to_upload_final_subset, how='inner', validate='many_to_one')
to_upload_final_tusb = tusb_to_upload.merge(to_upload_final_subset, how='inner', validate='many_to_one', on='flowcell_id')

In [194]:
to_upload_final_subset

,upload_sheet,sequencing_run,sequencing_date,sequencing_machine,sequencing_run,flowcell_position,flowcell_id,seqc_tube_tag,read_length
0,20250415_A00706_0926_BHVVTHDSXC.lanebarcode.html,0926,2025-04-15,Nova Seq 6000,0926,B,HVVTHDSXC,5OWD7,101
6,20250122_A00706_0912_BHVV2KDSXC.lanebarcode.html,0912,2025-01-22,Nova Seq 6000,0912,B,HVV2KDSXC,YDWCH,101
7,20250115_A00706_0909_BH5FG7DSX7.lanebarcode.html,0909,2025-01-15,Nova Seq 6000,0909,B,H5FG7DSX7,EDS7D,101
8,20250113_A00706_0907_BH5FG2DSX7.lanebarcode.html,0907,2025-01-13,Nova Seq 6000,0907,B,H5FG2DSX7,GCZGV,101
9,20250110_A00706_0904_BH5FC5DSX7.lanebarcode.html,0904,2025-01-10,Nova Seq 6000,0904,B,H5FC5DSX7,ACGTD,101
10,20250106_A00706_0900_BHCJY7DSX7.lanebarcode.html,0900,2025-01-06,Nova Seq 6000,0900,B,HCJY7DSX7,S84I4,101
11,20250103_A00706_0899_BH5CVCDSX7.lanebarcode.html,0899,2025-01-03,Nova Seq 6000,0899,B,H5CVCDSX7,RYLXU,101
12,20241129_A00706_0894_BH5FLYDSX7.lanebarcode.html,0894,2024-11-29,Nova Seq 6000,0894,B,H5FLYDSX7,CGJVB,101
13,20241106_A00706_0892_BH5CTKDSX7.lanebarcode.html,0892,2024-11-06,Nova Seq 6000,0892,B,H5CTKDSX7,3ANXN,101
14,240924_A00706_0882_BH5CW5DSX7_AMT6F.lanebarcod...,0882,2024-09-24,Nova Seq 6000,0882,B,H5CW5DSX7,AMT6F,101


In [195]:
tusb_to_upload

,ID,BSSHRunID,WorkingID,seqc_tube_tag,RunSheetID,sequencing_date,InstrumentID,flowcell_id,flowcell_position,SequencingOK,...,upload_sheet,run,date,machine,sequencing_run,side,flowcell,ttags,read_length,sequencing_machine
0,333,250415_A00706_0926_BHVVTHDSXC,250410_5OWD7_Xihan,5OWD7,250415_S4FCB,2025-04-15,A00706,HVVTHDSXC,B,True,...,20250415_A00706_0926_BHVVTHDSXC.lanebarcode.html,20250415_A00706_0926_BHVVTHDSXC,2025-04-15,A00706,0926,B,HVVTHDSXC,None,101,Nova Seq 6000
6,305,250122_A00706_0912_BHVV2KDSXC,250102_YDWCH_Suzan,YDWCH,250122_S4FCB,2025-01-22,A00706,HVV2KDSXC,B,True,...,20250122_A00706_0912_BHVV2KDSXC.lanebarcode.html,20250122_A00706_0912_BHVV2KDSXC,2025-01-22,A00706,0912,B,HVV2KDSXC,None,101,Nova Seq 6000
7,296,250115_A00706_0909_BH5FG7DSX7,250102_EDS7D_Suzan,EDS7D,250115_S4FCB,2025-01-15,A00706,H5FG7DSX7,B,True,...,20250115_A00706_0909_BH5FG7DSX7.lanebarcode.html,20250115_A00706_0909_BH5FG7DSX7,2025-01-15,A00706,0909,B,H5FG7DSX7,None,101,Nova Seq 6000
8,294,250113_A00706_0907_BH5FG2DSX7,241223_GCZGV_Suzan,GCZGV,250113_S4FCB,2025-01-13,A00706,H5FG2DSX7,B,True,...,20250113_A00706_0907_BH5FG2DSX7.lanebarcode.html,20250113_A00706_0907_BH5FG2DSX7,2025-01-13,A00706,0907,B,H5FG2DSX7,None,101,Nova Seq 6000
9,292,250110_A00706_0904_BH5FC5DSX7,241223_ACGTD_Suzan,ACGTD,250110_S4FCB,2025-01-10,A00706,H5FC5DSX7,B,True,...,20250110_A00706_0904_BH5FC5DSX7.lanebarcode.html,20250110_A00706_0904_BH5FC5DSX7,2025-01-10,A00706,0904,B,H5FC5DSX7,None,101,Nova Seq 6000
10,287,250106_A00706_0900_BHCJY7DSX7,241220_S84I4_Suzan,S84I4,250106_S4FCB,2025-01-06,A00706,HCJY7DSX7,B,True,...,20250106_A00706_0900_BHCJY7DSX7.lanebarcode.html,20250106_A00706_0900_BHCJY7DSX7,2025-01-06,A00706,0900,B,HCJY7DSX7,None,101,Nova Seq 6000
11,284,250103_A00706_0899_BH5CVCDSX7,241212_RYLXU_Suzan,RYLXU,250103_S4FCB,2025-01-03,A00706,H5CVCDSX7,B,True,...,20250103_A00706_0899_BH5CVCDSX7.lanebarcode.html,20250103_A00706_0899_BH5CVCDSX7,2025-01-03,A00706,0899,B,H5CVCDSX7,None,101,Nova Seq 6000
12,275,241129_A00706_0894_BH5FLYDSX7,241029_CGJVB_Maria,CGJVB,241129_S4FCB,2024-11-29,A00706,H5FLYDSX7,B,True,...,20241129_A00706_0894_BH5FLYDSX7.lanebarcode.html,20241129_A00706_0894_BH5FLYDSX7,2024-11-29,A00706,0894,B,H5FLYDSX7,None,101,Nova Seq 6000
13,268,241106_A00706_0892_BH5CTKDSX7,241023_3ANXN_Maria,3ANXN,241106_S4FCB,2024-11-06,A00706,H5CTKDSX7,B,True,...,20241106_A00706_0892_BH5CTKDSX7.lanebarcode.html,20241106_A00706_0892_BH5CTKDSX7,2024-11-06,A00706,0892,B,H5CTKDSX7,None,101,Nova Seq 6000
14,204,240924_A00706_0882_BH5CW5DSX7,240903_AMT6F_Maria,AMT6F,240924_S4FCB,2024-09-24,A00706,H5CW5DSX7,B,True,...,240924_A00706_0882_BH5CW5DSX7_AMT6F.lanebarcod...,240924_A00706_0882_BH5CW5DSX7_AMT6F,2024-09-24,A00706,0882,B,H5CW5DSX7,[AMT6F],101,Nova Seq 6000


In [196]:
tusb_to_upload['sequencing_run']

0     0926
6     0912
7     0909
8     0907
9     0904
10    0900
11    0899
12    0894
13    0892
14    0882
15    0880
16    0878
17    0874
18    0872
19    0871
21    0869
23    0864
35    0828
39    0929
40    0927
41    0925
45    0914
47    0910
48    0906
49    0908
50    0903
51    0901
52    0898
53    0893
54    0877
57    0873
58    0870
60    0860
61    0857
69    0829
72    0826
76    0811
Name: sequencing_run, dtype: object

In [197]:
seq_run = tusb_to_upload['sequencing_run']
tusb_to_upload = tusb_to_upload.drop(columns=['sequencing_run'], axis=1)
tusb_to_upload['sequencing_run'] = seq_run

In [198]:
tusb_to_upload['database_insert_by'] = 'julie.bitz-thorsen@sund.ku.dk'
from datetime import datetime, timezone

ts = datetime.now(timezone.utc)  
tusb_to_upload['database_insert_datetime_utc'] = ts
from uuid import uuid4
tusb_to_upload['upload_uuid'] = uuid4()
tusb_to_upload = tusb_to_upload.replace(["nan", "NaN", "NAN", "null", "NULL", "None", "none"], None)

In [199]:
tusb_to_upload.columns

Index(['ID', 'BSSHRunID', 'WorkingID', 'seqc_tube_tag', 'RunSheetID',
       'sequencing_date', 'InstrumentID', 'flowcell_id', 'flowcell_position',
       'SequencingOK', 'Notes by Julie', 'full_flowcell_id',
       'duplicate_flowcell', 'upload_sheet', 'run', 'date', 'machine', 'side',
       'flowcell', 'ttags', 'read_length', 'sequencing_machine',
       'sequencing_run', 'database_insert_by', 'database_insert_datetime_utc',
       'upload_uuid'],
      dtype='object')

In [200]:
top_unknown_barcodes.rename(columns={
    'Flowcell ID': 'flowcell_id'}, inplace=True)

In [201]:
tusb_to_upload.drop(columns=['ID', 
                             'BSSHRunID',
                             'WorkingID',
                             'RunSheetID',
                             'InstrumentID',
                             'Notes by Julie',
                             'SequencingOK',
                             'full_flowcell_id',
                             'duplicate_flowcell',
                             'run',
                             'date',
                             'machine',
                             'side',
                             'flowcell', 
                             'ttags'],
                             inplace=True, errors='ignore')


In [202]:
tusb_to_upload = tusb_to_upload.merge(top_unknown_barcodes, how='inner', validate='one_to_many', on='flowcell_id')

In [203]:
tusb_to_upload = tusb_to_upload.rename(columns=translator_tusb)

In [204]:
tusb_to_upload

,seqc_tube_tag,sequencing_date,flowcell_id,flowcell_position,upload_sheet,read_length,sequencing_machine,sequencing_run,database_insert_by,database_insert_datetime_utc,upload_uuid,unknown_barcode_lane,unknown_barcode_count,unknown_barcode_sequence
0,5OWD7,2025-04-15,HVVTHDSXC,B,20250415_A00706_0926_BHVVTHDSXC.lanebarcode.html,101,Nova Seq 6000,0926,julie.bitz-thorsen@sund.ku.dk,2025-12-22 10:33:40.379709+00:00,64e71280-c65e-4125-b6eb-4b18c8931473,1,995480,CTAAGTACGC+GCGGTATGTT
1,5OWD7,2025-04-15,HVVTHDSXC,B,20250415_A00706_0926_BHVVTHDSXC.lanebarcode.html,101,Nova Seq 6000,0926,julie.bitz-thorsen@sund.ku.dk,2025-12-22 10:33:40.379709+00:00,64e71280-c65e-4125-b6eb-4b18c8931473,1,714640,CTTACCGCAC+CGAGGTCTTA
2,5OWD7,2025-04-15,HVVTHDSXC,B,20250415_A00706_0926_BHVVTHDSXC.lanebarcode.html,101,Nova Seq 6000,0926,julie.bitz-thorsen@sund.ku.dk,2025-12-22 10:33:40.379709+00:00,64e71280-c65e-4125-b6eb-4b18c8931473,1,682880,GGCGCCATTG+CGTGCGCGGT
3,5OWD7,2025-04-15,HVVTHDSXC,B,20250415_A00706_0926_BHVVTHDSXC.lanebarcode.html,101,Nova Seq 6000,0926,julie.bitz-thorsen@sund.ku.dk,2025-12-22 10:33:40.379709+00:00,64e71280-c65e-4125-b6eb-4b18c8931473,1,512280,TTCTTAACCA+GGGCTGTAGA
4,5OWD7,2025-04-15,HVVTHDSXC,B,20250415_A00706_0926_BHVVTHDSXC.lanebarcode.html,101,Nova Seq 6000,0926,julie.bitz-thorsen@sund.ku.dk,2025-12-22 10:33:40.379709+00:00,64e71280-c65e-4125-b6eb-4b18c8931473,1,443020,NNNNNNNNNN+NNNNNNNNNN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,ZTM3M,2024-01-26,HTMW2DSX5,A,240126_A00706_0811_AHTMW2DSX5_603A.lanebarcode...,101,Nova Seq 6000,0811,julie.bitz-thorsen@sund.ku.dk,2025-12-22 10:33:40.379709+00:00,64e71280-c65e-4125-b6eb-4b18c8931473,4,1813580,TACCGAGGAT+CGGTCAGGGT
1476,ZTM3M,2024-01-26,HTMW2DSX5,A,240126_A00706_0811_AHTMW2DSX5_603A.lanebarcode...,101,Nova Seq 6000,0811,julie.bitz-thorsen@sund.ku.dk,2025-12-22 10:33:40.379709+00:00,64e71280-c65e-4125-b6eb-4b18c8931473,4,1745580,TGATGGCTAC+CGGTTGTTGG
1477,ZTM3M,2024-01-26,HTMW2DSX5,A,240126_A00706_0811_AHTMW2DSX5_603A.lanebarcode...,101,Nova Seq 6000,0811,julie.bitz-thorsen@sund.ku.dk,2025-12-22 10:33:40.379709+00:00,64e71280-c65e-4125-b6eb-4b18c8931473,4,1493260,GCTGACGTTG+CCTGCGAACA
1478,ZTM3M,2024-01-26,HTMW2DSX5,A,240126_A00706_0811_AHTMW2DSX5_603A.lanebarcode...,101,Nova Seq 6000,0811,julie.bitz-thorsen@sund.ku.dk,2025-12-22 10:33:40.379709+00:00,64e71280-c65e-4125-b6eb-4b18c8931473,4,1383760,GAACGCAATA+CCCGTAAGAT


In [205]:
tusb_to_upload.to_sql('top_unknown_seq_barcodes', ENGINE, if_exists='append', index=False)

480